
# ET Noise Budget Curves

These are plots of the noise budget curves and examples on how to work with them.

In [ ]:
# Load the necessary packages
import matplotlib.pyplot as plt
import gwinc 
import numpy as np
%matplotlib inline

# Baseline sensitivity

## ET High frequency

In [ ]:
budgetHF = gwinc.load_budget('ETHF') # load the budget from ETHF/ifo.yaml
tracesHF = budgetHF.run()  # compute the budget
fig = gwinc.plot_budget(tracesHF) # create the plot of the computed budget
plt.ylabel('Strain [$1 / \sqrt{\mathrm{Hz}}$]')   # add teh label
plt.ylim([5e-26, 1e-22])  # set the limits on y-axis for better presentation

## ET Low Frequency

In [ ]:
budgetLF = gwinc.load_budget('ETLF',freq=np.logspace(0,3,3000))  # freq defines the frequency range
tracesLF = budgetLF.run()
fig = gwinc.plot_budget(tracesLF)
plt.ylabel('Strain [$1 / \sqrt{\mathrm{Hz}}$]')
plt.ylim([5e-26, 1e-21])

## ET Total

This budget takes both ET LF and HF configuration and plots them together. It uses parameters from the corresponding yaml files. Noise computing and plotting can be configured in ET/__init__.py

In [ ]:
budgetET = gwinc.load_budget('ET')
tracesET = budgetET.run()
fig = gwinc.plot_budget(tracesET)
plt.ylabel('Strain [$1 / \sqrt{\mathrm{Hz}}$]')
plt.title('Curent model of ET NB ')
plt.ylim([5e-26, 1e-21])

# Advanced plotting different noises

Each noise contribution can be plotted separately. This section gives examples of that.

In [ ]:
### take ET LF as an example
budgetLF = gwinc.load_budget('ETLF',freq=np.logspace(0,3,3000))  
tracesLF = budgetLF.run()

tracesLF contains each contribution for each noise. They can be accessed indpependently

In [ ]:
print(tracesLF) # list all noises

In [ ]:
print(tracesLF.Quantum) # list contribution to the noise

In [ ]:
fig = gwinc.plot_budget(tracesLF.Quantum)  # plot all quantum noise contributions
plt.ylabel('Strain [$1 / \sqrt{\mathrm{Hz}}$]')
plt.title('Quantum noise contributions')
plt.ylim([5e-26, 1e-21])

Each individual noise contribution can also be accessed

In [ ]:
fig = gwinc.plot_noise(tracesLF.Quantum.Readout)  # plot an individual contribution
plt.ylabel('Strain [$1 / \sqrt{\mathrm{Hz}}$]')
plt.title('Quantum noise contributions')
plt.ylim([5e-26, 1e-21])

If you want to process the trace further, you can access the computed amplitude spectral density and create a plot by yourself.

In [ ]:
quantumReadout = tracesLF.Quantum.Readout.asd # take the amplitude spectral density of a given noise
freq = tracesLF.Quantum.Readout.freq  # take the frequency range of the computed plot
plt.loglog(freq, quantumReadout, c = 'brown', lw = 3)
plt.grid(which="both", c='gray', lw=0.5, alpha=0.5)
plt.ylabel('Strain [$1 / \sqrt{\mathrm{Hz}}$]')
plt.title('Readout loss contribution to quantum noise')
plt.ylim([5e-26, 1e-21])
plt.xlim([1,1000])

# Changing the parameters

pygwinc has two ways to change the parameters: directly from ipynb interface and through the yaml file. 

## Updating in the notebook

Change the parameter according to the yaml file following the hierarchy of the yaml file

```python
budgetLF.ifo.Seismic.Site= 'Terziet'        # changing the detector site
budgetLF.ifo.Materials.Substrate.Temp= 290  # changing the mirror temperature
budgetLF.ifo.Optics.SRM.Transmittance=0.05   # changing the SRM transmissivity
budgetLF.ifo.Squeezer.FilterCavity[0].L = 10000 # changing the length of the first filter cavity
```

## Updating via the file
If you want to change parameters in the yaml file,create a new file, e.g. ETLF_update.yaml with the following contents:
```
+inherit: '/Path/to/ETLF.yaml'

Optics:  
  SRM:  
    Tunephase: -0.6 # Updates SRM Tunephase

Squeezer:  
  FilterCavity:  
    - L: 1001 # Updates 1-st FC length  
    - L: 1002 # Updates 2-nd FC length
```

and then use this file as the source for loading the budget:

```python
budgetLF = gwinc.load_budget('ETLF/ETLF_update.yaml') # Loads updated NB
tracesLF = budgetLF.run() # Calculates updated noise traces of ETLF
```

In [ ]:
budgetLF = gwinc.load_budget('ETLF')  # load the budget
budgetLF.ifo.Optics.SRM.Transmittance=0.05   #change the SRM transmissivity
tracesLF = budgetLF.run()  # compute the baseline budget for comparison

fig = gwinc.plot_budget(tracesLF)
plt.ylabel('Strain [$1 / \sqrt{\mathrm{Hz}}$]')
plt.title('ETLF with modified SMR transmissivity')
plt.ylim([5e-26, 1e-21])

In [ ]:
### compare two different configurations (takes time to compute)

budgetHF = gwinc.load_budget('ETHF')  # load the budget
tracesHF = budgetHF.run()  # compute the baseline budget for comparison
suspensionTapered = tracesHF.SusThermal.asd
freq = tracesHF.SusThermal.freq  # take the frequency range of the computed plot

budgetHF.ifo.Suspension.Optcyl=0  #change the parameter, e.g. make the suspension fiber round instead of tapered
tracesHF = budgetHF.run()  # compute the new budget
suspensionRound = tracesHF.SusThermal.asd

plt.loglog(freq, suspensionTapered, c = 'brown', lw = 3, label = "Tapered")
plt.loglog(freq, suspensionRound, c = 'green', lw = 3, label = "Round")

plt.grid(which="both", c='gray', lw=0.5, alpha=0.5)
plt.ylabel('Strain [$1 / \sqrt{\mathrm{Hz}}$]')
plt.title('Suspension thermal noise comparison')
plt.ylim([5e-27, 1e-21])
plt.xlim([5,1000])
plt.legend()
plt.show()